# Structured R1 Evidence Consolidation

This notebook evaluates the **Structured R1 consolidation format**, the first structured-output extension of the Binary-only consolidation baseline.

The previous Binary-only experiment asked Qwen2.5-Omni to inspect the complete participant-centric evidence packet and return only:

```text
NORMAL
or
ANOMALOUS
```

That setup reached the thesis-reported Binary-only result:

| Format | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| Binary only | 72/100 | 54/100 | 95/100 | 100/100 | 80.25% |

Binary output, however, provides little visibility into **which evidence source the model appears to use** when making its decision.

Structured R1 therefore keeps the **same evidence packet and the same independent-normality decision policy**, but replaces the final binary-only output block with a categorical structured assessment.

## Structured R1 output

Before returning the final binary label, the model reports:

- `participation_assessment`
- `local_temporal_assessment`
- `global_temporal_assessment`
- `temporal_assessment`
- `semantic_assessment`
- `decisive_dimension`
- final `prediction`

The categorical values are deliberately constrained:

### Participation
- `VALID`
- `INVALID`

### Local / global / combined temporal evidence
- `NORMAL`
- `ANOMALOUS`
- `LIMITED`

### Semantic evidence
- `COMPATIBLE`
- `INCOMPATIBLE`
- `LIMITED`

### Decisive dimension
- `PARTICIPATION`
- `TEMPORAL`
- `SEMANTIC`
- `NONE`

The final task remains binary:

```text
NORMAL
or
ANOMALOUS
```

The structured fields are treated as **diagnostic reports of observable evidence utilisation**, not as guaranteed faithful traces of the model's hidden reasoning process.

## Controlled comparison with Binary-only

Structured R1 is designed as a controlled output-format intervention.

The notebook explicitly verifies that, relative to the Binary-only source experiment:

- the same 400 development cases are used;
- the same participant evidence is used;
- the same filtered turns are used;
- the same local temporal features are used;
- the same global temporal features are used;
- the same coarse semantic summaries are used;
- the same focused semantic summaries are used;
- the same frozen NORMAL temporal reference is used;
- the same Qwen2.5-Omni model is used;
- the same pre-output prompt text is retained.

The intentional changes are only:

1. Binary output block → Structured R1 output block.
2. `max_new_tokens` increases from 64 to 256 to allow the structured JSON response.

## Thesis-reported Structured R1 result

Structured R1 produces:

| Family | Correct |
|---|---:|
| NORMAL | **78/100** |
| LAG | **80/100** |
| WRONG PARTNER | **88/100** |
| SILENT PARTNER | **100/100** |
| **Overall** | **86.50%** |

The corresponding confusion matrix is:

```text
                Pred NORMAL   Pred ANOMALOUS
Gold NORMAL          78             22
Gold ANOMALOUS       32            268
```

This configuration is retained in the thesis as the main **auditable diagnostic consolidation format** and becomes the basis for the subsequent evidence-branch ablation analysis.

The remainder of this notebook also performs detailed R1 inspections, including:

- correct and missed LAG cases,
- correct and missed Wrong Partner cases,
- correct and false-positive NORMAL cases,
- Silent Partner behaviour,
- structured-field / final-label consistency,
- semantic-versus-temporal decision patterns,
- comparison with isolated semantic performance,
- inspection of `LIMITED` assessments,
- identification of anomalous decisions without an explicit failed dimension.

> **Reproducibility note:** all retained code cells, execution counts, metadata, and saved outputs are preserved exactly as executed. R2 and R3 have been removed from this public notebook so that it contains only the Structured R1 experiment and its diagnostics.

## 1. Install Dependencies

Install the Qwen2.5-Omni and evaluation dependencies used for structured text-only consolidation.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 149.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 49.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and Configure Artifact Paths

The structured-reasoning experiment operates on the same cached 400-case consolidation database used by the Binary-only source experiment.

Historical folder names are preserved exactly so that the original cached artifacts remain reproducible.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the Frozen NORMAL Temporal Reference

Structured R1 uses the same frozen NORMAL local-timing and global-alignment reference statistics as the Binary-only source experiment.

No LAG-specific reference profiles are introduced in R1.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and Audit the 400-Case Consolidation Database

The development set contains exactly:

- 100 NORMAL cases,
- 100 LAG cases,
- 100 WRONG PARTNER cases,
- 100 SILENT PARTNER cases.

The same structured evidence packet is used as in the Binary-only consolidation baseline.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-Level `speaks` Overview

The participation field is derived from the participant's final filtered VAD evidence and provides explicit speech grounding for the Silent Partner branch.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive Case Inspection

This utility exposes the complete structured evidence packet for manual auditing.

Any identifiers or gold metadata displayed here are for inspection only and are not included in the model-facing consolidation prompt.

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker

The consolidation reasoner is text-only at this stage: all audiovisual processing has already been converted into cached semantic, temporal, and participation evidence.

In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


## 8. Shared Qwen Inference Helpers

These helpers provide deterministic text generation and JSON parsing for the structured-reasoning experiment.

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


## 9. Shared Frozen NORMAL Reference Text

The reference-text construction is identical to the Binary-only source experiment and exposes only frozen NORMAL temporal statistics.

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

## 10. Shared Full-Semantics Input and Structured-Reasoning Helpers

The model-facing evidence projection is copied from the Binary-only consolidation notebook.

It passes exactly the same:

- participation fields,
- filtered turns,
- local temporal fields,
- global temporal fields,
- coarse semantic summaries,
- focused semantic summaries.

The only conceptual change is the output schema: instead of returning only a final label, the model must also emit categorical assessments for the evidence dimensions before the final binary prediction.

In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The model-facing input projection is copied from the original
# binary consolidation notebook:
#   - same participation fields
#   - same filtered turns
#   - same local temporal fields
#   - same global temporal fields
#   - same coarse summaries
#   - same focused summaries
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ]
            !=
            results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# Experiment — Structured R1

Structured R1 is derived directly from the thesis Binary-only source experiment:

```text
Binary-only source:
NORMAL          72 / 100
LAG             54 / 100
WRONG PARTNER   95 / 100
SILENT PARTNER 100 / 100
Accuracy        80.25%
```

The complete Binary-only prompt, evidence packet, model, and frozen NORMAL reference are retained.

Only the original binary `OUTPUT` block is replaced with the common Structured R1 schema.

This controlled design allows changes in prediction behaviour to be attributed to the **output / reasoning scaffold**, rather than to different evidence being supplied to the model.

In [ ]:

# ============================================================
# R1 CONFIGURATION
# FULL SEMANTICS + NORMAL REFERENCES ONLY + REASONING
#
# Exact source:
#   Experiment 2 — Independent Normality Requirements
#
# Only change:
#   binary OUTPUT block -> structured reasoning OUTPUT block
# ============================================================

R1_CONFIG = prepare_reasoning_experiment(
    experiment_version=(
        "reasoning_r1_full_semantics_"
        "normal_references_only"
    ),

    source_experiment_name=(
        "binary_only_consolidation_"
        "normal_definition_v2"
    ),

    experiment_title=(
        "R1 — Full Semantics, No LAG Profiles, "
        "Structured Reasoning"
    ),

    required_source_markers=[
        "NORMAL requires three independent properties",
        "FROZEN NORMAL LOCAL-TIMING REFERENCE",
        "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
        "Coarse semantic information:",
        "Focused semantic information:",
    ],

    forbidden_source_markers=[
        "FROZEN NON-NORMAL LOCAL-TIMING REFERENCE PROFILES",
        "Frozen LAG_2 local reference pattern",
        "Frozen LAG_3 local reference pattern",
        "TEMPORAL PROFILE COMPARISON",
        "ORDERED AND INDEPENDENT EVIDENCE ASSESSMENT",
    ],

    temporal_profiles_used=[
        "NORMAL",
    ],

    assessment_policy=(
        "Original Experiment 2 independent-normality "
        "requirements, unchanged."
    ),
)


R1 — Full Semantics, No LAG Profiles, Structured Reasoning — CONFIGURATION READY
Experiment version: reasoning_r1_full_semantics_normal_references_only
Source experiment: binary_only_consolidation_normal_definition_v2
Source prompt SHA256: ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace
Reasoning prompt SHA256: 7da269dce7d27e966cf1ecfa16543f7c55d5180b93d95cb33c3d44460bb5483f
Semantic input: coarse_and_focused
Focused summaries used: True
Temporal profiles: ['NORMAL']
Assessment policy: Original Experiment 2 independent-normality requirements, unchanged.
Only source-prompt change: binary OUTPUT block -> structured reasoning OUTPUT block
Example prompt characters: 25450
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/predictions_cache.json
Existing cache: True


In [ ]:

# ============================================================
# RUN R1
# ============================================================

R1_CACHE = run_reasoning_experiment(
    R1_CONFIG
)


Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/predictions_cache.json
Existing records: 400


reasoning_r1_full_semantics_normal_references_only:   0%|          | 0/400 [00:00<?, ?it/s]


R1 — Full Semantics, No LAG Profiles, Structured Reasoning — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/predictions_cache.json


In [ ]:

# ============================================================
# EVALUATE R1
# ============================================================

R1_EVALUATION = evaluate_reasoning_experiment(
    R1_CONFIG
)


R1 — Full Semantics, No LAG Profiles, Structured Reasoning — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8650
Balanced accuracy: 0.8367
ANOMALOUS precision: 0.9241
ANOMALOUS recall: 0.8933
ANOMALOUS F1: 0.9085
NORMAL recall / specificity: 0.7800
MCC: 0.6530
Matched source-group exact rate: 0.5800
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,78,22
Gold ANOMALOUS,32,268



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.709091,0.780000,0.742857,100.000
ANOMALOUS,0.924138,0.893333,0.908475,300.000
accuracy,0.865000,0.865000,0.865000,0.865
macro avg,0.816614,0.836667,0.825666,400.000
weighted avg,0.870376,0.865000,0.867070,400.000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,15,35,35,0.70,0.70,1.0
1,lag_3sec,50,50,0,5,45,45,0.90,0.90,1.0
2,normal,100,100,0,78,22,78,0.78,0.78,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,12,88,88,0.88,0.88,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,NORMAL,192
3,local_temporal_assessment,LIMITED,128
4,local_temporal_assessment,ANOMALOUS,80
5,global_temporal_assessment,ANOMALOUS,155
6,global_temporal_assessment,LIMITED,128
7,global_temporal_assessment,NORMAL,117
8,temporal_assessment,ANOMALOUS,155
9,temporal_assessment,LIMITED,128



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/prompt_diff_vs_source.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only/classification_errors.csv

All 400 cases produced valid binary p

In [ ]:
# ============================================================
# FORENSIC AUDIT:
# R1 vs ORIGINAL EXPERIMENT 2
#
# Verifies:
# 1. Historical source-template hash.
# 2. Source cache identity.
# 3. Same model and NORMAL references.
# 4. Exact same model-input payload for all 400 cases.
# 5. Exact reproduction of every original rendered prompt.
# 6. R1 differs only in the final OUTPUT section.
# 7. R1 cache contains exactly the prompts that were audited.
# ============================================================

from pathlib import Path

import json


# ============================================================
# PATHS
# ============================================================

SOURCE_EXPERIMENT_NAME = (
    "binary_only_consolidation_normal_definition_v2"
)


SOURCE_DIR = (
    OUT_DIR
    / SOURCE_EXPERIMENT_NAME
)


SOURCE_PROMPT_PATH = (
    SOURCE_DIR
    / "prompt_template.txt"
)


SOURCE_CACHE_PATH = (
    SOURCE_DIR
    / "predictions_cache.json"
)


R1_CACHE_PATH = (
    R1_CONFIG[
        "paths"
    ][
        "prediction_cache"
    ]
)


assert SOURCE_PROMPT_PATH.exists(), (
    f"Missing original prompt: {SOURCE_PROMPT_PATH}"
)


assert SOURCE_CACHE_PATH.exists(), (
    f"Missing original cache: {SOURCE_CACHE_PATH}"
)


assert R1_CACHE_PATH.exists(), (
    f"Missing R1 cache: {R1_CACHE_PATH}"
)


# ============================================================
# LOAD FILES
# ============================================================

source_prompt_template = (
    SOURCE_PROMPT_PATH.read_text(
        encoding="utf-8"
    )
)


source_cache = json.loads(
    SOURCE_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


r1_cache = json.loads(
    R1_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


r1_prompt_template = (
    R1_CONFIG[
        "reasoning_prompt_template"
    ]
)


# ============================================================
# 1. HISTORICAL EXPERIMENT-2 TEMPLATE HASH
#
# This is the hash printed and stored by the original run.
# ============================================================

EXPECTED_EXPERIMENT_2_PROMPT_SHA256 = (
    "ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace"
)


current_source_prompt_sha256 = (
    sha256_text(
        source_prompt_template
    )
)


assert (
    current_source_prompt_sha256
    == EXPECTED_EXPERIMENT_2_PROMPT_SHA256
), (
    "The current prompt_template.txt is not the historical "
    "Experiment 2 prompt."
)


assert (
    source_cache[
        "prompt_template_sha256"
    ]
    == EXPECTED_EXPERIMENT_2_PROMPT_SHA256
), (
    "The original prediction cache was not generated with "
    "the expected Experiment 2 prompt."
)


assert (
    R1_CONFIG[
        "source_prompt_sha256"
    ]
    == EXPECTED_EXPERIMENT_2_PROMPT_SHA256
), (
    "R1 did not load the expected Experiment 2 source prompt."
)


assert (
    R1_CONFIG[
        "source_prompt_template"
    ]
    == source_prompt_template
), (
    "The source prompt held by R1 differs from the saved "
    "Experiment 2 prompt."
)


assert (
    r1_cache[
        "source_prompt_sha256"
    ]
    == EXPECTED_EXPERIMENT_2_PROMPT_SHA256
), (
    "The completed R1 cache is not tied to the expected "
    "Experiment 2 source prompt."
)


# ============================================================
# 2. SAME MODEL AND SAME FROZEN NORMAL REFERENCES
# ============================================================

assert (
    source_cache[
        "model_id"
    ]
    == r1_cache[
        "model_id"
    ]
    == MODEL_ID
), (
    "Model mismatch between Experiment 2 and R1."
)


assert (
    source_cache[
        "normal_reference_sha256"
    ]
    == r1_cache[
        "normal_reference_sha256"
    ]
), (
    "Frozen NORMAL reference mismatch."
)


# ============================================================
# 3. TEMPLATE-LEVEL DIFFERENCE
#
# Everything before the old OUTPUT block must be identical.
# ============================================================

assert source_prompt_template.endswith(
    OLD_BINARY_OUTPUT_BLOCK
)


assert r1_prompt_template.endswith(
    STRUCTURED_REASONING_OUTPUT_BLOCK
)


source_pre_output_template = (
    source_prompt_template[
        :-len(
            OLD_BINARY_OUTPUT_BLOCK
        )
    ]
)


r1_pre_output_template = (
    r1_prompt_template[
        :-len(
            STRUCTURED_REASONING_OUTPUT_BLOCK
        )
    ]
)


assert (
    source_pre_output_template
    == r1_pre_output_template
), (
    "R1 changed prompt text before the OUTPUT section."
)


reconstructed_original_template = (
    r1_pre_output_template
    + OLD_BINARY_OUTPUT_BLOCK
)


assert (
    reconstructed_original_template
    == source_prompt_template
), (
    "The original Experiment 2 prompt cannot be exactly "
    "reconstructed from the R1 prompt."
)


# ============================================================
# 4. ALL 400 CASES MUST EXIST IN BOTH CACHES
# ============================================================

source_record_ids = set(
    source_cache[
        "records"
    ].keys()
)


r1_record_ids = set(
    r1_cache[
        "records"
    ].keys()
)


database_case_ids = {
    str(
        case[
            "case_id"
        ]
    )

    for case in consolidation_cases
}


assert len(
    database_case_ids
) == 400


assert (
    source_record_ids
    == database_case_ids
), (
    "Original Experiment 2 cache does not contain exactly "
    "the current 400 cases."
)


assert (
    r1_record_ids
    == database_case_ids
), (
    "R1 cache does not contain exactly the same 400 cases."
)


# ============================================================
# 5. CASE-LEVEL FORENSIC AUDIT
# ============================================================

source_prompt_matches = 0
payload_matches = 0
pre_output_matches = 0
r1_cache_prompt_matches = 0
r1_cache_payload_matches = 0

audit_failures = []


SOURCE_RENDERED_OUTPUT_MARKER = """
============================================================
OUTPUT
============================================================
""".strip()


R1_RENDERED_OUTPUT_MARKER = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================
""".strip()


for case in sorted(
    consolidation_cases,
    key=lambda item: str(
        item[
            "case_id"
        ]
    ),
):

    case_id = str(
        case[
            "case_id"
        ]
    )


    original_record = (
        source_cache[
            "records"
        ][case_id]
    )


    r1_record = (
        r1_cache[
            "records"
        ][case_id]
    )


    # --------------------------------------------------------
    # Recreate the exact original Experiment 2 prompt using
    # the reasoning notebook's current payload builder.
    #
    # If the copied builder differs from the original builder,
    # the original cache hashes will expose the difference.
    # --------------------------------------------------------

    recreated_original_prompt, original_payload = (
        build_binary_prompt_from_template(
            case,
            source_prompt_template,
        )
    )


    recreated_original_prompt_sha256 = (
        sha256_text(
            recreated_original_prompt
        )
    )


    original_payload_sha256 = (
        sha256_text(
            canonical_json(
                original_payload
            )
        )
    )


    # --------------------------------------------------------
    # Recreate the actual R1 prompt.
    # --------------------------------------------------------

    recreated_r1_prompt, r1_payload = (
        build_binary_prompt_from_template(
            case,
            r1_prompt_template,
        )
    )


    recreated_r1_prompt_sha256 = (
        sha256_text(
            recreated_r1_prompt
        )
    )


    r1_payload_sha256 = (
        sha256_text(
            canonical_json(
                r1_payload
            )
        )
    )


    case_errors = []


    # Exact original rendered prompt versus original cache.
    if (
        recreated_original_prompt_sha256
        ==
        original_record[
            "prompt_sha256"
        ]
    ):

        source_prompt_matches += 1

    else:

        case_errors.append(
            "original_prompt_hash_mismatch"
        )


    # Exact payload versus original cache.
    if (
        original_payload_sha256
        ==
        original_record[
            "input_payload_sha256"
        ]
    ):

        payload_matches += 1

    else:

        case_errors.append(
            "original_payload_hash_mismatch"
        )


    # R1 must use exactly the same payload.
    if (
        original_payload_sha256
        !=
        r1_payload_sha256
    ):

        case_errors.append(
            "r1_payload_differs_from_original"
        )


    # R1 prompt stored in cache must equal the prompt recreated now.
    if (
        recreated_r1_prompt_sha256
        ==
        r1_record[
            "prompt_sha256"
        ]
    ):

        r1_cache_prompt_matches += 1

    else:

        case_errors.append(
            "r1_cache_prompt_hash_mismatch"
        )


    if (
        r1_payload_sha256
        ==
        r1_record[
            "input_payload_sha256"
        ]
    ):

        r1_cache_payload_matches += 1

    else:

        case_errors.append(
            "r1_cache_payload_hash_mismatch"
        )


    # --------------------------------------------------------
    # Compare the fully rendered case prompts before OUTPUT.
    # --------------------------------------------------------

    assert (
        SOURCE_RENDERED_OUTPUT_MARKER
        in recreated_original_prompt
    )


    assert (
        R1_RENDERED_OUTPUT_MARKER
        in recreated_r1_prompt
    )


    original_pre_output = (
        recreated_original_prompt.split(
            SOURCE_RENDERED_OUTPUT_MARKER,
            1,
        )[0]
    )


    r1_pre_output = (
        recreated_r1_prompt.split(
            R1_RENDERED_OUTPUT_MARKER,
            1,
        )[0]
    )


    if (
        original_pre_output
        == r1_pre_output
    ):

        pre_output_matches += 1

    else:

        case_errors.append(
            "rendered_pre_output_prompt_mismatch"
        )


    if case_errors:

        audit_failures.append({
            "case_id": case_id,
            "errors": case_errors,
        })


# ============================================================
# 6. FINAL ASSERTIONS
# ============================================================

assert not audit_failures, (
    "R1 forensic audit failed. First failures:\n"
    f"{audit_failures[:10]}"
)


assert source_prompt_matches == 400
assert payload_matches == 400
assert pre_output_matches == 400
assert r1_cache_prompt_matches == 400
assert r1_cache_payload_matches == 400


# Expected and intentional generation-length difference.
source_max_new_tokens = {
    record[
        "max_new_tokens"
    ]

    for record in source_cache[
        "records"
    ].values()
}


r1_max_new_tokens = {
    record[
        "max_new_tokens"
    ]

    for record in r1_cache[
        "records"
    ].values()
}


assert source_max_new_tokens == {
    64
}


assert r1_max_new_tokens == {
    256
}


# ============================================================
# REPORT
# ============================================================

print("=" * 88)
print("R1 FORENSIC AUDIT PASSED")
print("=" * 88)

print(
    "Historical Experiment 2 prompt SHA256:",
    current_source_prompt_sha256,
)

print(
    "Original rendered prompts reproduced exactly:",
    f"{source_prompt_matches}/400",
)

print(
    "Original input payloads reproduced exactly:",
    f"{payload_matches}/400",
)

print(
    "R1 and Experiment 2 payloads identical:",
    "400/400",
)

print(
    "Pre-output prompt text identical:",
    f"{pre_output_matches}/400",
)

print(
    "R1 cached prompts verified:",
    f"{r1_cache_prompt_matches}/400",
)

print(
    "R1 cached payloads verified:",
    f"{r1_cache_payload_matches}/400",
)

print(
    "Same model:",
    MODEL_ID,
)

print(
    "Same NORMAL-reference SHA256:",
    source_cache[
        "normal_reference_sha256"
    ],
)

print(
    "Intentional differences only:",
)

print(
    "  1. Binary OUTPUT block -> structured reasoning OUTPUT block"
)

print(
    "  2. max_new_tokens: 64 -> 256"
)

R1 FORENSIC AUDIT PASSED
Historical Experiment 2 prompt SHA256: ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace
Original rendered prompts reproduced exactly: 400/400
Original input payloads reproduced exactly: 400/400
R1 and Experiment 2 payloads identical: 400/400
Pre-output prompt text identical: 400/400
R1 cached prompts verified: 400/400
R1 cached payloads verified: 400/400
Same model: Qwen/Qwen2.5-Omni-7B
Same NORMAL-reference SHA256: 8ada07c4141d0c3db070cbbbd9081aa7d0113001a7196cfcb8168f5b6f40e23a
Intentional differences only:
  1. Binary OUTPUT block -> structured reasoning OUTPUT block
  2. max_new_tokens: 64 -> 256


In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD R1 CASE-LEVEL RESULTS
# Run this after the "EVALUATE R1" cell.
# ============================================================

if "R1_EVALUATION" in globals():

    r1_df = (
        R1_EVALUATION[
            "results_df"
        ].copy()
    )

elif "R1_CONFIG" in globals():

    r1_df = pd.read_csv(
        R1_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the R1 configuration and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    r1_df[
        column
    ] = (
        r1_df[
            column
        ]
        .astype(
            "string"
        )
        .str.strip()
        .str.upper()
    )


r1_df[
    "case_family"
] = (
    r1_df[
        "case_family"
    ]
    .astype(
        "string"
    )
    .str.strip()
    .str.lower()
)


r1_df[
    "case_variant"
] = (
    r1_df[
        "case_variant"
    ]
    .astype(
        "string"
    )
    .str.strip()
    .str.lower()
)


if (
    "valid_prediction"
    not in r1_df.columns
):

    r1_df[
        "valid_prediction"
    ] = (
        r1_df[
            "prediction"
        ].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if (
    "correct"
    not in r1_df.columns
):

    r1_df[
        "correct"
    ] = (
        r1_df[
            "valid_prediction"
        ]
        &
        (
            r1_df[
                "gold_label"
            ]
            ==
            r1_df[
                "prediction"
            ]
        )
    )


# ============================================================
# EXPECTED VALUES FOR EVERY STRUCTURED FIELD
# ============================================================

ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "semantic_assessment": [
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def frequency_table(
    subset,
    field,
    expected_values=None,
):

    values = (
        subset[
            field
        ]
        .fillna(
            "MISSING"
        )
        .astype(
            str
        )
        .str.upper()
    )


    if expected_values is None:

        expected_values = sorted(
            values.unique().tolist()
        )


    ordered_values = list(
        dict.fromkeys(
            list(
                expected_values
            )
            +
            [
                "MISSING",
            ]
        )
    )


    counts = (
        values
        .value_counts(
            dropna=False
        )
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    denominator = len(
        subset
    )


    table = pd.DataFrame({
        "assessment_field": (
            field
        ),

        "assessment_value": (
            counts.index
        ),

        "count": (
            counts.values
        ),
    })


    table[
        "percentage"
    ] = (
        100.0
        *
        table[
            "count"
        ]
        /
        denominator

        if denominator

        else 0.0
    )


    table[
        "percentage"
    ] = (
        table[
            "percentage"
        ].round(
            2
        )
    )


    return table


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_r1_subset(
    subset,
    title,
):

    subset = subset.copy()


    print(
        "=" * 90
    )

    print(
        title
    )

    print(
        "=" * 90
    )


    if subset.empty:

        print(
            "No cases in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(
            subset
        ),

        "gold_NORMAL": int(
            (
                subset[
                    "gold_label"
                ]
                == "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset[
                    "gold_label"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct": int(
            subset[
                "correct"
            ].sum()
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    print(
        "\nCASE VARIANT BREAKDOWN"
    )


    variant_counts = (
        subset[
            "case_variant"
        ]
        .fillna(
            "MISSING"
        )
        .value_counts()
        .rename_axis(
            "case_variant"
        )
        .reset_index(
            name="count"
        )
    )


    variant_counts[
        "percentage"
    ] = (
        100.0
        *
        variant_counts[
            "count"
        ]
        /
        len(
            subset
        )
    ).round(
        2
    )


    display(
        variant_counts
    )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(

        [
            frequency_table(
                subset,
                field,
                expected_values,
            )

            for (
                field,
                expected_values,
            )
            in ASSESSMENT_LEVELS.items()
        ],

        ignore_index=True,
    )


    display(
        assessment_table
    )


    print(
        "\nLOCAL × GLOBAL TEMPORAL ASSESSMENT"
    )


    display(
        pd.crosstab(

            subset[
                "local_temporal_assessment"
            ].fillna(
                "MISSING"
            ),

            subset[
                "global_temporal_assessment"
            ].fillna(
                "MISSING"
            ),

            margins=True,
        )
    )


    print(
        "\nCOMBINED TEMPORAL × DECISIVE DIMENSION"
    )


    display(
        pd.crosstab(

            subset[
                "temporal_assessment"
            ].fillna(
                "MISSING"
            ),

            subset[
                "decisive_dimension"
            ].fillna(
                "MISSING"
            ),

            margins=True,
        )
    )


    print(
        "\nSEMANTIC ASSESSMENT × DECISIVE DIMENSION"
    )


    display(
        pd.crosstab(

            subset[
                "semantic_assessment"
            ].fillna(
                "MISSING"
            ),

            subset[
                "decisive_dimension"
            ].fillna(
                "MISSING"
            ),

            margins=True,
        )
    )


    return subset


# ============================================================
# INITIAL SANITY CHECK
# ============================================================

print(
    "R1 cases loaded:",
    len(
        r1_df
    ),
)


display(
    pd.crosstab(
        r1_df[
            "case_family"
        ],
        r1_df[
            "prediction"
        ],
        margins=True,
    )
)

R1 cases loaded: 400


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,80,20,100
normal,22,78,100
silent_partner,100,0,100
wrong_partner,88,12,100
All,290,110,400


In [ ]:
# ============================================================
# LAG — CORRECTLY PREDICTED AS ANOMALOUS
# ============================================================

lag_correct_anomalous = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


inspect_r1_subset(
    lag_correct_anomalous,
    (
        "R1 — LAG CORRECTLY DETECTED "
        "AS ANOMALOUS"
    ),
)

R1 — LAG CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,80,0,80,0,80,80



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_3sec,45,56.25
1,lag_2sec,35,43.75



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,TEMPORAL,All
temporal_assessment,,
ANOMALOUS,75,75
LIMITED,5,5
All,80,80



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,TEMPORAL,All
semantic_assessment,,
COMPATIBLE,74,74
LIMITED,6,6
All,80,80


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
200,consolidation_lag_2sec_000,heldout_source_000,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5128,9.7692,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
202,consolidation_lag_3sec_002,heldout_source_002,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5178,8.7036,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
204,consolidation_lag_3sec_004,heldout_source_004,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5203,8.6800,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
205,consolidation_lag_3sec_005,heldout_source_005,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5020,8.5223,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
206,consolidation_lag_3sec_006,heldout_source_006,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5077,8.7029,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,consolidation_lag_3sec_094,heldout_source_094,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5015,8.7028,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
296,consolidation_lag_3sec_096,heldout_source_096,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5117,8.6436,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
297,consolidation_lag_3sec_097,heldout_source_097,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5135,8.6736,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
298,consolidation_lag_3sec_098,heldout_source_098,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,...,5137,8.7224,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False


In [ ]:
# ============================================================
# LAG — MISSED AND PREDICTED AS NORMAL
# ============================================================

lag_missed_normal = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


inspect_r1_subset(
    lag_missed_normal,
    (
        "R1 — LAG MISSED AND "
        "PREDICTED AS NORMAL"
    ),
)

R1 — LAG MISSED AND PREDICTED AS NORMAL

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,20,0,20,20,0,0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_2sec,15,75.0
1,lag_3sec,5,25.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
temporal_assessment,,
NORMAL,20,20
All,20,20



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
semantic_assessment,,
COMPATIBLE,20,20
All,20,20


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
201,consolidation_lag_2sec_001,heldout_source_001,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,4990,8.5125,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
203,consolidation_lag_2sec_003,heldout_source_003,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5082,8.6244,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
209,consolidation_lag_2sec_009,heldout_source_009,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5163,8.6052,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
211,consolidation_lag_2sec_011,heldout_source_011,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5245,8.6869,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
216,consolidation_lag_3sec_016,heldout_source_016,lag,lag_3sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5183,8.6417,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
225,consolidation_lag_2sec_025,heldout_source_025,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5192,8.6665,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
231,consolidation_lag_2sec_031,heldout_source_031,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5110,8.6411,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
236,consolidation_lag_2sec_036,heldout_source_036,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5065,8.7043,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
247,consolidation_lag_2sec_047,heldout_source_047,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5011,8.5843,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
248,consolidation_lag_2sec_048,heldout_source_048,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5007,8.6787,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False


In [ ]:
# ============================================================
# WRONG PARTNER — CORRECTLY PREDICTED AS ANOMALOUS
# ============================================================

wrong_partner_correct_anomalous = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


inspect_r1_subset(
    wrong_partner_correct_anomalous,
    (
        "R1 — WRONG PARTNER CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)

R1 — WRONG PARTNER CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,88,0,88,0,88,88



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,wrong_partner,88,100.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,TEMPORAL,All
temporal_assessment,,,
ANOMALOUS,2,59,61
LIMITED,13,7,20
NORMAL,7,0,7
All,22,66,88



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,TEMPORAL,All
semantic_assessment,,,
COMPATIBLE,0,37,37
INCOMPATIBLE,7,4,11
LIMITED,15,25,40
All,22,66,88


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
100,consolidation_wrong_partner_000,heldout_source_000,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5183,8.6439,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
101,consolidation_wrong_partner_001,heldout_source_001,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,4977,8.6313,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
102,consolidation_wrong_partner_002,heldout_source_002,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,...,5101,8.6806,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
103,consolidation_wrong_partner_003,heldout_source_003,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5115,8.6498,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
104,consolidation_wrong_partner_004,heldout_source_004,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5160,8.6665,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,consolidation_wrong_partner_095,heldout_source_095,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5158,8.6840,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
196,consolidation_wrong_partner_096,heldout_source_096,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,...,5011,8.5949,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
197,consolidation_wrong_partner_097,heldout_source_097,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5150,8.6091,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
198,consolidation_wrong_partner_098,heldout_source_098,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5168,8.6765,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False


In [ ]:
# ============================================================
# WRONG PARTNER — MISSED AND PREDICTED AS NORMAL
# ============================================================

wrong_partner_missed_normal = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


inspect_r1_subset(
    wrong_partner_missed_normal,
    (
        "R1 — WRONG PARTNER MISSED "
        "AND PREDICTED AS NORMAL"
    ),
)

R1 — WRONG PARTNER MISSED AND PREDICTED AS NORMAL

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,12,0,12,12,0,0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,wrong_partner,12,100.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
temporal_assessment,,
NORMAL,12,12
All,12,12



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
semantic_assessment,,
COMPATIBLE,12,12
All,12,12


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
106,consolidation_wrong_partner_006,heldout_source_006,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5070,8.6665,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
109,consolidation_wrong_partner_009,heldout_source_009,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5014,8.6184,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
110,consolidation_wrong_partner_010,heldout_source_010,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5203,8.7201,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
121,consolidation_wrong_partner_021,heldout_source_021,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5298,8.6544,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
122,consolidation_wrong_partner_022,heldout_source_022,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5300,8.6110,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
127,consolidation_wrong_partner_027,heldout_source_027,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5243,8.5722,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
136,consolidation_wrong_partner_036,heldout_source_036,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5049,8.7355,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
171,consolidation_wrong_partner_071,heldout_source_071,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5221,8.6979,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
172,consolidation_wrong_partner_072,heldout_source_072,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5093,8.7804,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
176,consolidation_wrong_partner_076,heldout_source_076,wrong_partner,wrong_partner,ANOMALOUS,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5057,8.6766,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False


In [ ]:
# ============================================================
# SILENT PARTNER — CORRECTLY PREDICTED AS ANOMALOUS
# ============================================================

silent_partner_correct_anomalous = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


inspect_r1_subset(
    silent_partner_correct_anomalous,
    (
        "R1 — SILENT PARTNER CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)

R1 — SILENT PARTNER CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,100,0,100,0,100,100



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,silent_partner,100,100.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,PARTICIPATION,All
temporal_assessment,,
LIMITED,100,100
All,100,100



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,PARTICIPATION,All
semantic_assessment,,
COMPATIBLE,11,11
LIMITED,89,89
All,100,100


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
300,consolidation_silent_partner_000,heldout_source_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,5014,8.6815,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
301,consolidation_silent_partner_001,heldout_source_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4866,8.6560,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
302,consolidation_silent_partner_002,heldout_source_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,5060,8.6165,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
303,consolidation_silent_partner_003,heldout_source_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4958,8.6660,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
304,consolidation_silent_partner_004,heldout_source_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,5006,8.6748,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,consolidation_silent_partner_095,heldout_source_095,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4845,8.7568,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
396,consolidation_silent_partner_096,heldout_source_096,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4933,8.6898,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
397,consolidation_silent_partner_097,heldout_source_097,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4975,8.7020,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False
398,consolidation_silent_partner_098,heldout_source_098,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,...,4830,8.6374,"```json\n{\n ""participation_assessment"": ""INV...",None,True,True,False,False,False,False


In [ ]:
silent_partner_missed = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


print(
    "Silent-partner cases predicted NORMAL:",
    len(
        silent_partner_missed
    ),
)

Silent-partner cases predicted NORMAL: 0


In [ ]:
# ============================================================
# NORMAL — CORRECTLY PREDICTED AS NORMAL
# ============================================================

normal_correct_normal = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "NORMAL"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


inspect_r1_subset(
    normal_correct_normal,
    (
        "R1 — NORMAL CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)

R1 — NORMAL CORRECTLY PREDICTED AS NORMAL

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,78,78,0,78,0,78



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,normal,78,100.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
temporal_assessment,,
NORMAL,78,78
All,78,78



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,SEMANTIC,All
semantic_assessment,,
COMPATIBLE,78,78
All,78,78


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
1,consolidation_normal_001,heldout_source_001,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,4990,8.6250,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
4,consolidation_normal_004,heldout_source_004,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5195,8.6472,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
5,consolidation_normal_005,heldout_source_005,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5061,8.5796,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
6,consolidation_normal_006,heldout_source_006,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5074,8.5586,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
7,consolidation_normal_007,heldout_source_007,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5449,8.6346,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,consolidation_normal_093,heldout_source_093,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5066,8.7067,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
94,consolidation_normal_094,heldout_source_094,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5042,8.6513,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
97,consolidation_normal_097,heldout_source_097,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5136,8.6325,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False
98,consolidation_normal_098,heldout_source_098,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5156,8.6307,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,True,False,False,False,False


In [ ]:
# ============================================================
# NORMAL — FALSELY PREDICTED AS ANOMALOUS
# ============================================================

normal_false_anomalous = r1_df[
    (
        r1_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        r1_df[
            "gold_label"
        ]
        == "NORMAL"
    )
    &
    (
        r1_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


inspect_r1_subset(
    normal_false_anomalous,
    (
        "R1 — NORMAL FALSELY "
        "PREDICTED AS ANOMALOUS"
    ),
)

R1 — NORMAL FALSELY PREDICTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,correct
0,22,22,0,0,22,0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,normal,22,100.0



ASSESSMENT BREAKDOWN


...


COMBINED TEMPORAL × DECISIVE DIMENSION


decisive_dimension,TEMPORAL,All
temporal_assessment,,
ANOMALOUS,19,19
LIMITED,3,3
All,22,22



SEMANTIC ASSESSMENT × DECISIVE DIMENSION


decisive_dimension,TEMPORAL,All
semantic_assessment,,
COMPATIBLE,18,18
LIMITED,4,4
All,22,22


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
0,consolidation_normal_000,heldout_source_000,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5128,8.6146,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
2,consolidation_normal_002,heldout_source_002,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5157,8.6807,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
3,consolidation_normal_003,heldout_source_003,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5074,8.6503,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
9,consolidation_normal_009,heldout_source_009,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5143,8.6866,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
17,consolidation_normal_017,heldout_source_017,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5251,8.6876,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
18,consolidation_normal_018,heldout_source_018,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,4994,8.6719,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
27,consolidation_normal_027,heldout_source_027,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5060,8.5956,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
33,consolidation_normal_033,heldout_source_033,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5072,8.6663,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
35,consolidation_normal_035,heldout_source_035,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,4931,8.5766,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False
41,consolidation_normal_041,heldout_source_041,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5182,8.6520,"```json\n{\n ""participation_assessment"": ""VAL...",None,True,False,False,False,False,False


In [ ]:
# ============================================================
# FINAL LABEL / INTERMEDIATE ASSESSMENT CONSISTENCY
# ============================================================

r1_consistency = (
    r1_df.copy()
)


# ============================================================
# IDENTIFY FAILED DIMENSIONS
# ============================================================

r1_consistency[
    "participation_failure"
] = (
    r1_consistency[
        "participation_assessment"
    ]
    == "INVALID"
)


r1_consistency[
    "temporal_failure"
] = (
    r1_consistency[
        "temporal_assessment"
    ]
    == "ANOMALOUS"
)


r1_consistency[
    "semantic_failure"
] = (
    r1_consistency[
        "semantic_assessment"
    ]
    == "INCOMPATIBLE"
)


r1_consistency[
    "has_any_failure"
] = (
    r1_consistency[
        [
            "participation_failure",
            "temporal_failure",
            "semantic_failure",
        ]
    ].any(
        axis=1
    )
)


# ============================================================
# HARD FINAL-LABEL CONSISTENCY
#
# NORMAL:
#   no failed independent dimension
#
# ANOMALOUS:
#   at least one failed independent dimension
# ============================================================

r1_consistency[
    "label_supported_by_assessments"
] = (

    (
        (
            r1_consistency[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            ~r1_consistency[
                "has_any_failure"
            ]
        )
    )

    |

    (
        (
            r1_consistency[
                "prediction"
            ]
            == "ANOMALOUS"
        )
        &
        r1_consistency[
            "has_any_failure"
        ]
    )
)


# ============================================================
# STRICT DECISIVE-DIMENSION CONSISTENCY
#
# NORMAL -> NONE
#
# ANOMALOUS:
#   PARTICIPATION -> participation INVALID
#   TEMPORAL      -> temporal ANOMALOUS
#   SEMANTIC      -> semantic INCOMPATIBLE
# ============================================================

r1_consistency[
    "decisive_dimension_supported"
] = False


# NORMAL should have NONE.

r1_consistency.loc[

    (
        r1_consistency[
            "prediction"
        ]
        == "NORMAL"
    )
    &
    (
        r1_consistency[
            "decisive_dimension"
        ]
        == "NONE"
    ),

    "decisive_dimension_supported",

] = True


# ANOMALOUS because of participation.

r1_consistency.loc[

    (
        r1_consistency[
            "prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_consistency[
            "decisive_dimension"
        ]
        == "PARTICIPATION"
    )
    &
    r1_consistency[
        "participation_failure"
    ],

    "decisive_dimension_supported",

] = True


# ANOMALOUS because of temporal evidence.

r1_consistency.loc[

    (
        r1_consistency[
            "prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_consistency[
            "decisive_dimension"
        ]
        == "TEMPORAL"
    )
    &
    r1_consistency[
        "temporal_failure"
    ],

    "decisive_dimension_supported",

] = True


# ANOMALOUS because of semantic evidence.

r1_consistency.loc[

    (
        r1_consistency[
            "prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        r1_consistency[
            "decisive_dimension"
        ]
        == "SEMANTIC"
    )
    &
    r1_consistency[
        "semantic_failure"
    ],

    "decisive_dimension_supported",

] = True


# ============================================================
# CONSISTENCY SUMMARY BY CASE FAMILY AND PREDICTION
# ============================================================

consistency_summary = (

    r1_consistency

    .groupby(
        [
            "case_family",
            "prediction",
        ],
        dropna=False,
    )

    .agg(

        cases=(
            "case_id",
            "size",
        ),

        label_supported_count=(
            "label_supported_by_assessments",
            "sum",
        ),

        decisive_supported_count=(
            "decisive_dimension_supported",
            "sum",
        ),
    )

    .reset_index()
)


consistency_summary[
    "label_supported_pct"
] = (
    100.0
    *
    consistency_summary[
        "label_supported_count"
    ]
    /
    consistency_summary[
        "cases"
    ]
).round(
    2
)


consistency_summary[
    "decisive_supported_pct"
] = (
    100.0
    *
    consistency_summary[
        "decisive_supported_count"
    ]
    /
    consistency_summary[
        "cases"
    ]
).round(
    2
)


print(
    "CONSISTENCY SUMMARY"
)

display(
    consistency_summary
)


# ============================================================
# FINAL-LABEL CONFLICTS
# ============================================================

print(
    "\nFINAL LABEL CONFLICTS"
)


label_conflicts = r1_consistency[
    ~r1_consistency[
        "label_supported_by_assessments"
    ]
].copy()


display(
    label_conflicts[
        [
            "case_id",
            "case_family",
            "case_variant",
            "gold_label",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    ]
)


# ============================================================
# DECISIVE-DIMENSION CONFLICTS
# ============================================================

print(
    "\nDECISIVE-DIMENSION CONFLICTS"
)


decisive_conflicts = r1_consistency[
    ~r1_consistency[
        "decisive_dimension_supported"
    ]
].copy()


display(
    decisive_conflicts[
        [
            "case_id",
            "case_family",
            "case_variant",
            "gold_label",
            "prediction",
            "participation_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    ]
)


# ============================================================
# OBSERVED TEMPORAL COMBINATION PATTERNS
#
# This does not declare combinations correct or incorrect.
# The temporal policy is qualitative, not deterministic.
# ============================================================

print(
    "\nOBSERVED LOCAL/GLOBAL/COMBINED TEMPORAL PATTERNS"
)


temporal_combination_patterns = (

    r1_consistency

    .groupby(
        [
            "case_family",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
        ],
        dropna=False,
    )

    .size()

    .reset_index(
        name="count"
    )

    .sort_values(
        [
            "case_family",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


display(
    temporal_combination_patterns
)

CONSISTENCY SUMMARY


,case_family,prediction,cases,label_supported_count,decisive_supported_count,label_supported_pct,decisive_supported_pct
0,lag,ANOMALOUS,80,75,75,93.75,93.75
1,lag,NORMAL,20,20,0,100.0,0.00
2,normal,ANOMALOUS,22,19,19,86.36,86.36
3,normal,NORMAL,78,78,0,100.0,0.00
4,silent_partner,ANOMALOUS,100,100,100,100.0,100.00
5,wrong_partner,ANOMALOUS,88,66,66,75.0,75.00
6,wrong_partner,NORMAL,12,12,0,100.0,0.00



FINAL LABEL CONFLICTS


,case_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension
65,consolidation_normal_065,normal,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
82,consolidation_normal_082,normal,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
86,consolidation_normal_086,normal,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
102,consolidation_wrong_partner_002,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC
111,consolidation_wrong_partner_011,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
114,consolidation_wrong_partner_014,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC
124,consolidation_wrong_partner_024,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
129,consolidation_wrong_partner_029,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC
132,consolidation_wrong_partner_032,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,NORMAL,NORMAL,NORMAL,LIMITED,SEMANTIC
135,consolidation_wrong_partner_035,wrong_partner,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC



DECISIVE-DIMENSION CONFLICTS


,case_id,case_family,case_variant,gold_label,prediction,participation_assessment,temporal_assessment,semantic_assessment,decisive_dimension
1,consolidation_normal_001,normal,normal,NORMAL,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
4,consolidation_normal_004,normal,normal,NORMAL,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
5,consolidation_normal_005,normal,normal,NORMAL,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
6,consolidation_normal_006,normal,normal,NORMAL,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
7,consolidation_normal_007,normal,normal,NORMAL,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
...,...,...,...,...,...,...,...,...,...
282,consolidation_lag_2sec_082,lag,lag_2sec,ANOMALOUS,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
286,consolidation_lag_2sec_086,lag,lag_2sec,ANOMALOUS,ANOMALOUS,VALID,LIMITED,COMPATIBLE,TEMPORAL
295,consolidation_lag_3sec_095,lag,lag_3sec,ANOMALOUS,NORMAL,VALID,NORMAL,COMPATIBLE,SEMANTIC
298,consolidation_lag_3sec_098,lag,lag_3sec,ANOMALOUS,ANOMALOUS,VALID,LIMITED,COMPATIBLE,TEMPORAL



OBSERVED LOCAL/GLOBAL/COMBINED TEMPORAL PATTERNS


,case_family,local_temporal_assessment,global_temporal_assessment,temporal_assessment,count
2,lag,NORMAL,ANOMALOUS,ANOMALOUS,39
0,lag,ANOMALOUS,ANOMALOUS,ANOMALOUS,36
3,lag,NORMAL,NORMAL,NORMAL,20
1,lag,LIMITED,LIMITED,LIMITED,5
7,normal,NORMAL,NORMAL,NORMAL,78
4,normal,ANOMALOUS,ANOMALOUS,ANOMALOUS,10
6,normal,NORMAL,ANOMALOUS,ANOMALOUS,9
5,normal,LIMITED,LIMITED,LIMITED,3
8,silent_partner,LIMITED,LIMITED,LIMITED,100
9,wrong_partner,ANOMALOUS,ANOMALOUS,ANOMALOUS,34


In [ ]:
# ============================================================
# WRONG PARTNER:
# SEMANTIC ASSESSMENT × DECISIVE DIMENSION × PREDICTION
# ============================================================

wp_r1 = r1_df[
    r1_df["case_family"] == "wrong_partner"
].copy()


print("ALL WRONG-PARTNER CASES")

display(
    pd.crosstab(
        [
            wp_r1["prediction"],
            wp_r1["semantic_assessment"],
        ],
        wp_r1["decisive_dimension"],
        margins=True,
    )
)

In [ ]:
# ============================================================
# WRONG PARTNER:
# SEMANTIC ASSESSMENT × DECISIVE DIMENSION × PREDICTION
# ============================================================

wp_r1 = r1_df[
    r1_df["case_family"] == "wrong_partner"
].copy()


print("ALL WRONG-PARTNER CASES")

display(
    pd.crosstab(
        [
            wp_r1["prediction"],
            wp_r1["semantic_assessment"],
        ],
        wp_r1["decisive_dimension"],
        margins=True,
    )
)

ALL WRONG-PARTNER CASES


decisive_dimension              SEMANTIC  TEMPORAL  All
prediction semantic_assessment                         
ANOMALOUS  COMPATIBLE                  0        37   37
           INCOMPATIBLE                7         4   11
           LIMITED                    15        25   40
NORMAL     COMPATIBLE                 12         0   12
All                                   34        66  100

In [ ]:
# ============================================================
# CASES WHERE SEMANTIC IS DECLARED DECISIVE
# ============================================================

wp_semantic_decisive = wp_r1[
    wp_r1["decisive_dimension"] == "SEMANTIC"
].copy()


display(
    wp_semantic_decisive[
        [
            "case_id",
            "prediction",
            "semantic_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "decisive_dimension",
        ]
    ].sort_values(
        [
            "prediction",
            "semantic_assessment",
        ]
    )
)

,case_id,prediction,semantic_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,decisive_dimension
117,consolidation_wrong_partner_017,ANOMALOUS,INCOMPATIBLE,NORMAL,NORMAL,NORMAL,SEMANTIC
130,consolidation_wrong_partner_030,ANOMALOUS,INCOMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,SEMANTIC
150,consolidation_wrong_partner_050,ANOMALOUS,INCOMPATIBLE,NORMAL,NORMAL,NORMAL,SEMANTIC
151,consolidation_wrong_partner_051,ANOMALOUS,INCOMPATIBLE,NORMAL,NORMAL,NORMAL,SEMANTIC
153,consolidation_wrong_partner_053,ANOMALOUS,INCOMPATIBLE,NORMAL,NORMAL,NORMAL,SEMANTIC
164,consolidation_wrong_partner_064,ANOMALOUS,INCOMPATIBLE,NORMAL,NORMAL,NORMAL,SEMANTIC
168,consolidation_wrong_partner_068,ANOMALOUS,INCOMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,SEMANTIC
102,consolidation_wrong_partner_002,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC
114,consolidation_wrong_partner_014,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC
129,consolidation_wrong_partner_029,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC


In [ ]:
# ============================================================
# STRICT SEMANTIC-DECISIVE CONSISTENCY
#
# If SEMANTIC is decisive:
# - ANOMALOUS should normally require INCOMPATIBLE
# - NORMAL should normally require COMPATIBLE
# ============================================================

wp_semantic_decisive[
    "strict_semantic_consistency"
] = (

    (
        (
            wp_semantic_decisive["prediction"]
            == "ANOMALOUS"
        )
        &
        (
            wp_semantic_decisive["semantic_assessment"]
            == "INCOMPATIBLE"
        )
    )

    |

    (
        (
            wp_semantic_decisive["prediction"]
            == "NORMAL"
        )
        &
        (
            wp_semantic_decisive["semantic_assessment"]
            == "COMPATIBLE"
        )
    )
)


print("SEMANTIC-DECISIVE CONSISTENCY")

display(
    wp_semantic_decisive[
        "strict_semantic_consistency"
    ].value_counts(
        dropna=False
    ).rename_axis(
        "consistent"
    ).reset_index(
        name="count"
    )
)


print("\nINCONSISTENT CASES")

display(
    wp_semantic_decisive[
        ~wp_semantic_decisive[
            "strict_semantic_consistency"
        ]
    ][
        [
            "case_id",
            "prediction",
            "semantic_assessment",
            "temporal_assessment",
            "decisive_dimension",
        ]
    ]
)

SEMANTIC-DECISIVE CONSISTENCY


,consistent,count
0,True,19
1,False,15



INCONSISTENT CASES


,case_id,prediction,semantic_assessment,temporal_assessment,decisive_dimension
102,consolidation_wrong_partner_002,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
114,consolidation_wrong_partner_014,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
129,consolidation_wrong_partner_029,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
132,consolidation_wrong_partner_032,ANOMALOUS,LIMITED,NORMAL,SEMANTIC
135,consolidation_wrong_partner_035,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
145,consolidation_wrong_partner_045,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
148,consolidation_wrong_partner_048,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
154,consolidation_wrong_partner_054,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
155,consolidation_wrong_partner_055,ANOMALOUS,LIMITED,LIMITED,SEMANTIC
156,consolidation_wrong_partner_056,ANOMALOUS,LIMITED,LIMITED,SEMANTIC


In [ ]:
# ============================================================
# LOAD SEMANTIC-ONLY RESULTS
# NORMAL vs WRONG PARTNER
# COARSE + FULL FOCUSED SUMMARIES
# ============================================================

from pathlib import Path

import pandas as pd
from IPython.display import display


SEMANTIC_ONLY_PREDICTIONS_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "normal_vs_wrong_partner_"
    "coarse_plus_full_focused_original_prompt_v1/"
    "predictions.csv"
)


assert SEMANTIC_ONLY_PREDICTIONS_PATH.exists(), (
    "Semantic-only predictions CSV not found:\n"
    f"{SEMANTIC_ONLY_PREDICTIONS_PATH}"
)


semantic_only_df = pd.read_csv(
    SEMANTIC_ONLY_PREDICTIONS_PATH
)


# Normalize identifiers and labels.

semantic_only_df["case_id"] = (
    semantic_only_df["case_id"]
    .astype(str)
    .str.strip()
)


semantic_only_df["case_family"] = (
    semantic_only_df["case_family"]
    .astype(str)
    .str.strip()
    .str.lower()
)


semantic_only_df["gold_label"] = (
    semantic_only_df["gold_label"]
    .astype(str)
    .str.strip()
    .str.upper()
)


semantic_only_df["predicted_label"] = (
    semantic_only_df["predicted_label"]
    .astype(str)
    .str.strip()
    .str.upper()
)


print(
    "Loaded semantic-only cases:",
    len(semantic_only_df),
)

print(
    "Unique case IDs:",
    semantic_only_df["case_id"].nunique(),
)

print(
    "Path:",
    SEMANTIC_ONLY_PREDICTIONS_PATH,
)


assert len(semantic_only_df) == 200, (
    "Expected 200 semantic-only cases."
)

assert semantic_only_df["case_id"].is_unique, (
    "Semantic-only case IDs are not unique."
)

assert semantic_only_df["predicted_label"].isin(
    [
        "NORMAL",
        "ANOMALOUS",
    ]
).all(), (
    "Invalid semantic-only predictions found."
)


print("\nSEMANTIC-ONLY CONFUSION MATRIX")

display(
    pd.crosstab(
        semantic_only_df["gold_label"],
        semantic_only_df["predicted_label"],
        rownames=["Gold"],
        colnames=["Prediction"],
        margins=True,
    )
)

Loaded semantic-only cases: 200
Unique case IDs: 200
Path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/normal_vs_wrong_partner_coarse_plus_full_focused_original_prompt_v1/predictions.csv

SEMANTIC-ONLY CONFUSION MATRIX


Prediction,ANOMALOUS,NORMAL,All
Gold,,,
ANOMALOUS,98,2,100
NORMAL,21,79,100
All,119,81,200


In [ ]:
# ============================================================
# SAME-CASE COMPARISON:
# SEMANTIC-ONLY vs R1 STRUCTURED SEMANTIC ASSESSMENT
# ============================================================

# Ensure that R1 identifiers use the same format.

r1_df["case_id"] = (
    r1_df["case_id"]
    .astype(str)
    .str.strip()
)


r1_df["case_family"] = (
    r1_df["case_family"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# PREPARE SEMANTIC-ONLY RESULTS
# ============================================================

semantic_only_compare = (
    semantic_only_df[
        [
            "case_id",
            "case_family",
            "gold_label",
            "predicted_label",
        ]
    ]
    .copy()
    .rename(
        columns={
            "case_family": (
                "semantic_only_case_family"
            ),

            "gold_label": (
                "semantic_only_gold_label"
            ),

            "predicted_label": (
                "semantic_only_prediction"
            ),
        }
    )
)


# ============================================================
# PREPARE R1 RESULTS
# Keep only NORMAL and WRONG PARTNER cases.
# ============================================================

r1_wp_compare = (
    r1_df[
        r1_df["case_family"].isin(
            [
                "normal",
                "wrong_partner",
            ]
        )
    ][
        [
            "case_id",
            "case_family",
            "gold_label",
            "prediction",
            "participation_assessment",
            "semantic_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "decisive_dimension",
        ]
    ]
    .copy()
    .rename(
        columns={
            "gold_label": (
                "r1_gold_label"
            ),

            "prediction": (
                "r1_prediction"
            ),
        }
    )
)


assert len(r1_wp_compare) == 200, (
    "Expected 200 R1 NORMAL + WRONG PARTNER cases."
)

assert r1_wp_compare["case_id"].is_unique, (
    "R1 case IDs are not unique."
)


# ============================================================
# MERGE THE SAME CASES
# ============================================================

semantic_vs_r1 = (
    semantic_only_compare
    .merge(
        r1_wp_compare,
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("case_id")
    .reset_index(drop=True)
)


print(
    "Matched cases:",
    len(semantic_vs_r1),
)


assert len(semantic_vs_r1) == 200, (
    "The semantic-only and R1 experiments did not match "
    "on all 200 case IDs."
)


# Confirm that both experiments use the same case family.

assert (
    semantic_vs_r1[
        "semantic_only_case_family"
    ]
    ==
    semantic_vs_r1[
        "case_family"
    ]
).all(), (
    "Case-family mismatch between experiments."
)


# Confirm that both experiments use the same gold label.

assert (
    semantic_vs_r1[
        "semantic_only_gold_label"
    ]
    ==
    semantic_vs_r1[
        "r1_gold_label"
    ]
).all(), (
    "Gold-label mismatch between experiments."
)


# Remove duplicate validation columns.

semantic_vs_r1 = (
    semantic_vs_r1
    .drop(
        columns=[
            "semantic_only_case_family",
            "semantic_only_gold_label",
        ]
    )
)


# ============================================================
# MAIN COMPARISON
# ============================================================

print(
    "\nSEMANTIC-ONLY PREDICTION "
    "× R1 SEMANTIC ASSESSMENT"
)


display(
    pd.crosstab(
        [
            semantic_vs_r1[
                "case_family"
            ],

            semantic_vs_r1[
                "semantic_only_prediction"
            ],
        ],

        semantic_vs_r1[
            "semantic_assessment"
        ],

        rownames=[
            "Case family",
            "Semantic-only prediction",
        ],

        colnames=[
            "R1 semantic assessment",
        ],

        margins=True,
    )
)

Matched cases: 200

SEMANTIC-ONLY PREDICTION × R1 SEMANTIC ASSESSMENT


R1 semantic assessment                  COMPATIBLE  INCOMPATIBLE  LIMITED  All
Case family   Semantic-only prediction                                        
normal        ANOMALOUS                         17             0        4   21
              NORMAL                            79             0        0   79
wrong_partner ANOMALOUS                         47            11       40   98
              NORMAL                             2             0        0    2
All                                            145            11       44  200

In [ ]:
# ============================================================
# WRONG PARTNER ONLY:
# SEMANTIC-ONLY PREDICTION vs R1 SEMANTIC ASSESSMENT
# ============================================================

wrong_partner_semantic_comparison = (
    semantic_vs_r1[
        semantic_vs_r1[
            "case_family"
        ]
        == "wrong_partner"
    ]
    .copy()
)


print(
    "Wrong-partner cases:",
    len(
        wrong_partner_semantic_comparison
    ),
)


display(
    pd.crosstab(
        wrong_partner_semantic_comparison[
            "semantic_only_prediction"
        ],

        wrong_partner_semantic_comparison[
            "semantic_assessment"
        ],

        rownames=[
            "Semantic-only prediction",
        ],

        colnames=[
            "R1 semantic assessment",
        ],

        margins=True,
    )
)

Wrong-partner cases: 100


R1 semantic assessment,COMPATIBLE,INCOMPATIBLE,LIMITED,All
Semantic-only prediction,,,,
ANOMALOUS,47,11,40,98
NORMAL,2,0,0,2
All,49,11,40,100


In [ ]:
# ============================================================
# POSSIBLE SEMANTIC SUPPRESSION / INTERFERENCE
#
# Wrong partner correctly detected by semantic-only experiment,
# but R1 semantic assessment says COMPATIBLE.
# ============================================================

semantic_suppression_candidates = (
    semantic_vs_r1[
        (
            semantic_vs_r1[
                "case_family"
            ]
            == "wrong_partner"
        )
        &
        (
            semantic_vs_r1[
                "semantic_only_prediction"
            ]
            == "ANOMALOUS"
        )
        &
        (
            semantic_vs_r1[
                "semantic_assessment"
            ]
            == "COMPATIBLE"
        )
    ]
    .copy()
)


print(
    "Semantic suppression/interference candidates:",
    len(
        semantic_suppression_candidates
    ),
)


display(
    semantic_suppression_candidates[
        [
            "case_id",
            "semantic_only_prediction",
            "semantic_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "decisive_dimension",
            "r1_prediction",
        ]
    ].sort_values(
        [
            "r1_prediction",
            "temporal_assessment",
            "case_id",
        ]
    )
)

Semantic suppression/interference candidates: 47


,case_id,semantic_only_prediction,semantic_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,decisive_dimension,r1_prediction
101,consolidation_wrong_partner_001,ANOMALOUS,COMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
103,consolidation_wrong_partner_003,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
104,consolidation_wrong_partner_004,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
105,consolidation_wrong_partner_005,ANOMALOUS,COMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
115,consolidation_wrong_partner_015,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
116,consolidation_wrong_partner_016,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
118,consolidation_wrong_partner_018,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
119,consolidation_wrong_partner_019,ANOMALOUS,COMPATIBLE,ANOMALOUS,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
123,consolidation_wrong_partner_023,ANOMALOUS,COMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS
125,consolidation_wrong_partner_025,ANOMALOUS,COMPATIBLE,NORMAL,ANOMALOUS,ANOMALOUS,TEMPORAL,ANOMALOUS


In [ ]:
# ============================================================
# WRONG PARTNERS DETECTED BY SEMANTIC-ONLY EXPERIMENT:
# WHAT DID R1 DO WITH THE SAME CASES?
# ============================================================

semantic_only_wp_true_positives = (
    semantic_vs_r1[
        (
            semantic_vs_r1[
                "case_family"
            ]
            == "wrong_partner"
        )
        &
        (
            semantic_vs_r1[
                "semantic_only_prediction"
            ]
            == "ANOMALOUS"
        )
    ]
    .copy()
)


print(
    "Wrong partners detected as ANOMALOUS "
    "by semantic-only experiment:",
    len(
        semantic_only_wp_true_positives
    ),
)


print(
    "\nR1 SEMANTIC ASSESSMENT"
)

display(
    semantic_only_wp_true_positives[
        "semantic_assessment"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "semantic_assessment"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nR1 TEMPORAL ASSESSMENT"
)

display(
    semantic_only_wp_true_positives[
        "temporal_assessment"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "temporal_assessment"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nR1 FINAL PREDICTION"
)

display(
    semantic_only_wp_true_positives[
        "r1_prediction"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "r1_prediction"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nR1 SEMANTIC × TEMPORAL ASSESSMENT"
)

display(
    pd.crosstab(
        semantic_only_wp_true_positives[
            "semantic_assessment"
        ],

        semantic_only_wp_true_positives[
            "temporal_assessment"
        ],

        margins=True,
    )
)

Wrong partners detected as ANOMALOUS by semantic-only experiment: 98

R1 SEMANTIC ASSESSMENT


,semantic_assessment,count
0,COMPATIBLE,47
1,LIMITED,40
2,INCOMPATIBLE,11



R1 TEMPORAL ASSESSMENT


,temporal_assessment,count
0,ANOMALOUS,60
1,LIMITED,20
2,NORMAL,18



R1 FINAL PREDICTION


,r1_prediction,count
0,ANOMALOUS,87
1,NORMAL,11



R1 SEMANTIC × TEMPORAL ASSESSMENT


temporal_assessment,ANOMALOUS,LIMITED,NORMAL,All
semantic_assessment,,,,
COMPATIBLE,29,7,11,47
INCOMPATIBLE,6,0,5,11
LIMITED,25,13,2,40
All,60,20,18,98


In [ ]:
# ============================================================
# WRONG PARTNERS DETECTED BY SEMANTIC-ONLY:
# CLASSIFY HOW R1 REACHED ITS FINAL DECISION
# ============================================================

comparison_subset = (
    semantic_only_wp_true_positives.copy()
)


comparison_subset[
    "r1_decision_pattern"
] = "OTHER"


# ------------------------------------------------------------
# 1. CLEAN TEMPORAL RESCUE
#
# R1 semantics do not identify incompatibility,
# but temporal evidence is explicitly anomalous.
# ------------------------------------------------------------

comparison_subset.loc[
    (
        comparison_subset[
            "r1_prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        comparison_subset[
            "semantic_assessment"
        ]
        .isin([
            "COMPATIBLE",
            "LIMITED",
        ])
    )
    &
    (
        comparison_subset[
            "temporal_assessment"
        ]
        == "ANOMALOUS"
    ),
    "r1_decision_pattern",
] = "CLEAN_TEMPORAL_RESCUE"


# ------------------------------------------------------------
# 2. CLEAN SEMANTIC DETECTION
#
# Semantic incompatibility explicitly supports anomaly,
# while temporal evidence is not anomalous.
# ------------------------------------------------------------

comparison_subset.loc[
    (
        comparison_subset[
            "r1_prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        comparison_subset[
            "semantic_assessment"
        ]
        == "INCOMPATIBLE"
    )
    &
    (
        comparison_subset[
            "temporal_assessment"
        ]
        .isin([
            "NORMAL",
            "LIMITED",
        ])
    ),
    "r1_decision_pattern",
] = "CLEAN_SEMANTIC_DETECTION"


# ------------------------------------------------------------
# 3. BOTH SEMANTIC AND TEMPORAL SUPPORT ANOMALY
# ------------------------------------------------------------

comparison_subset.loc[
    (
        comparison_subset[
            "r1_prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        comparison_subset[
            "semantic_assessment"
        ]
        == "INCOMPATIBLE"
    )
    &
    (
        comparison_subset[
            "temporal_assessment"
        ]
        == "ANOMALOUS"
    ),
    "r1_decision_pattern",
] = "BOTH_SUPPORT_ANOMALY"


# ------------------------------------------------------------
# 4. UNSUPPORTED / INTERNALLY INCONSISTENT ANOMALY
#
# Final anomaly, but neither explicit semantic nor
# explicit temporal assessment supports failure.
# ------------------------------------------------------------

comparison_subset.loc[
    (
        comparison_subset[
            "r1_prediction"
        ]
        == "ANOMALOUS"
    )
    &
    (
        comparison_subset[
            "semantic_assessment"
        ]
        .isin([
            "COMPATIBLE",
            "LIMITED",
        ])
    )
    &
    (
        comparison_subset[
            "temporal_assessment"
        ]
        .isin([
            "NORMAL",
            "LIMITED",
        ])
    ),
    "r1_decision_pattern",
] = "UNSUPPORTED_ANOMALOUS_DECISION"


# ------------------------------------------------------------
# 5. MISSED BY R1
# ------------------------------------------------------------

comparison_subset.loc[
    (
        comparison_subset[
            "r1_prediction"
        ]
        == "NORMAL"
    ),
    "r1_decision_pattern",
] = "MISSED_BY_R1"


print(
    "R1 DECISION PATTERNS FOR THE 98 WRONG PARTNERS "
    "DETECTED BY SEMANTIC-ONLY"
)


pattern_counts = (
    comparison_subset[
        "r1_decision_pattern"
    ]
    .value_counts()
    .rename_axis(
        "decision_pattern"
    )
    .reset_index(
        name="count"
    )
)


pattern_counts[
    "percentage"
] = (
    100.0
    *
    pattern_counts[
        "count"
    ]
    /
    len(
        comparison_subset
    )
).round(
    2
)


display(
    pattern_counts
)

R1 DECISION PATTERNS FOR THE 98 WRONG PARTNERS DETECTED BY SEMANTIC-ONLY


,decision_pattern,count,percentage
0,CLEAN_TEMPORAL_RESCUE,54,55.10
1,UNSUPPORTED_ANOMALOUS_DECISION,22,22.45
2,MISSED_BY_R1,11,11.22
3,BOTH_SUPPORT_ANOMALY,6,6.12
4,CLEAN_SEMANTIC_DETECTION,5,5.10


In [ ]:
# ============================================================
# SHOW INTERNALLY INCONSISTENT ANOMALOUS DECISIONS
# ============================================================

unsupported_anomalies = comparison_subset[
    comparison_subset[
        "r1_decision_pattern"
    ]
    == "UNSUPPORTED_ANOMALOUS_DECISION"
].copy()


print(
    "Unsupported anomalous decisions:",
    len(
        unsupported_anomalies
    ),
)


display(
    unsupported_anomalies[
        [
            "case_id",
            "semantic_only_prediction",
            "semantic_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "decisive_dimension",
            "r1_prediction",
        ]
    ].sort_values(
        [
            "semantic_assessment",
            "temporal_assessment",
            "case_id",
        ]
    )
)

Unsupported anomalous decisions: 22


,case_id,semantic_only_prediction,semantic_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,decisive_dimension,r1_prediction
111,consolidation_wrong_partner_011,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
124,consolidation_wrong_partner_024,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
138,consolidation_wrong_partner_038,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
144,consolidation_wrong_partner_044,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
178,consolidation_wrong_partner_078,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
182,consolidation_wrong_partner_082,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
184,consolidation_wrong_partner_084,ANOMALOUS,COMPATIBLE,LIMITED,LIMITED,LIMITED,TEMPORAL,ANOMALOUS
102,consolidation_wrong_partner_002,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC,ANOMALOUS
114,consolidation_wrong_partner_014,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC,ANOMALOUS
129,consolidation_wrong_partner_029,ANOMALOUS,LIMITED,LIMITED,LIMITED,LIMITED,SEMANTIC,ANOMALOUS


In [ ]:
# ============================================================
# ALL CASES WITH SEMANTIC ASSESSMENT = LIMITED
# FINAL PREDICTION BREAKDOWN
# ============================================================

semantic_limited_cases = r1_df[
    r1_df["semantic_assessment"] == "LIMITED"
].copy()


print(
    "Total semantic LIMITED cases:",
    len(semantic_limited_cases),
)


semantic_limited_prediction_counts = (
    semantic_limited_cases["prediction"]
    .value_counts(dropna=False)
    .reindex(
        [
            "NORMAL",
            "ANOMALOUS",
        ],
        fill_value=0,
    )
    .rename_axis("prediction")
    .reset_index(name="count")
)


semantic_limited_prediction_counts["percentage"] = (
    100.0
    * semantic_limited_prediction_counts["count"]
    / len(semantic_limited_cases)
).round(2)


display(
    semantic_limited_prediction_counts
)

Total semantic LIMITED cases: 139


,prediction,count,percentage
0,NORMAL,0,0.0
1,ANOMALOUS,139,100.0


In [ ]:
# ============================================================
# SEMANTIC LIMITED:
# FINAL PREDICTION BY CASE FAMILY
# ============================================================

display(
    pd.crosstab(
        semantic_limited_cases["case_family"],
        semantic_limited_cases["prediction"],
        rownames=["Case family"],
        colnames=["Final prediction"],
        margins=True,
    )
)

Final prediction,ANOMALOUS,All
Case family,,
lag,6,6
normal,4,4
silent_partner,89,89
wrong_partner,40,40
All,139,139


In [ ]:
# ============================================================
# ALL CASES WITH COMBINED TEMPORAL ASSESSMENT = LIMITED
# FINAL PREDICTION BREAKDOWN
# ============================================================

temporal_limited_cases = r1_df[
    r1_df["temporal_assessment"] == "LIMITED"
].copy()


print(
    "Total temporal LIMITED cases:",
    len(temporal_limited_cases),
)


temporal_limited_prediction_counts = (
    temporal_limited_cases["prediction"]
    .value_counts(dropna=False)
    .reindex(
        [
            "NORMAL",
            "ANOMALOUS",
        ],
        fill_value=0,
    )
    .rename_axis("prediction")
    .reset_index(name="count")
)


temporal_limited_prediction_counts["percentage"] = (
    100.0
    * temporal_limited_prediction_counts["count"]
    / len(temporal_limited_cases)
).round(2)


display(
    temporal_limited_prediction_counts
)

Total temporal LIMITED cases: 128


,prediction,count,percentage
0,NORMAL,0,0.0
1,ANOMALOUS,128,100.0


In [ ]:
# ============================================================
# TEMPORAL LIMITED:
# FINAL PREDICTION BY CASE FAMILY
# ============================================================

display(
    pd.crosstab(
        temporal_limited_cases["case_family"],
        temporal_limited_cases["prediction"],
        rownames=["Case family"],
        colnames=["Final prediction"],
        margins=True,
    )
)

Final prediction,ANOMALOUS,All
Case family,,
lag,5,5
normal,3,3
silent_partner,100,100
wrong_partner,20,20
All,128,128


In [ ]:
# ============================================================
# ANOMALOUS PREDICTIONS WITHOUT AN EXPLICIT FAILED DIMENSION
# ============================================================

unsupported_all_r1 = r1_df[
    (
        r1_df["prediction"]
        == "ANOMALOUS"
    )
    &
    (
        r1_df["participation_assessment"]
        != "INVALID"
    )
    &
    (
        r1_df["temporal_assessment"]
        != "ANOMALOUS"
    )
    &
    (
        r1_df["semantic_assessment"]
        != "INCOMPATIBLE"
    )
].copy()


print(
    "All R1 anomalous predictions without "
    "an explicit failed dimension:",
    len(unsupported_all_r1),
)


display(
    pd.crosstab(
        unsupported_all_r1["case_family"],
        unsupported_all_r1["decisive_dimension"],
        margins=True,
    )
)


display(
    unsupported_all_r1[
        [
            "case_id",
            "case_family",
            "gold_label",
            "prediction",
            "participation_assessment",
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
            "decisive_dimension",
        ]
    ].sort_values(
        [
            "case_family",
            "semantic_assessment",
            "temporal_assessment",
            "case_id",
        ]
    )
)

All R1 anomalous predictions without an explicit failed dimension: 30


decisive_dimension,SEMANTIC,TEMPORAL,All
case_family,,,
lag,0,5,5
normal,0,3,3
wrong_partner,15,7,22
All,15,15,30


,case_id,case_family,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension
233,consolidation_lag_2sec_033,lag,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
286,consolidation_lag_2sec_086,lag,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
275,consolidation_lag_3sec_075,lag,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
298,consolidation_lag_3sec_098,lag,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
299,consolidation_lag_3sec_099,lag,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
65,consolidation_normal_065,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
82,consolidation_normal_082,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
86,consolidation_normal_086,normal,NORMAL,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
111,consolidation_wrong_partner_011,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL
124,consolidation_wrong_partner_024,wrong_partner,ANOMALOUS,ANOMALOUS,VALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,TEMPORAL


# Structured R1 Conclusion

Structured R1 changes the observable behaviour of the unified reasoner despite receiving the same underlying evidence as the Binary-only source experiment.

The final class-specific performance is:

```text
NORMAL          78 / 100
LAG             80 / 100
WRONG PARTNER   88 / 100
SILENT PARTNER 100 / 100

Overall accuracy: 86.50%
```

Compared with Binary-only, Structured R1 improves the balance between NORMAL preservation and LAG detection while retaining perfect Silent Partner performance.

More importantly, the structured categorical fields make it possible to inspect **how semantic, temporal, and participation evidence are represented in the model's observable output**.

The subsequent diagnostics reveal that:

- correct LAG decisions are usually associated with anomalous temporal assessments;
- many correctly detected Wrong Partner cases are not marked semantically `INCOMPATIBLE`;
- temporal evidence frequently carries Wrong Partner decisions;
- `LIMITED` assessments can still lead directly to `ANOMALOUS`;
- structured fields and final predictions are inspectable, but they should not be interpreted as proof of independent or faithful internal reasoning.

For this reason, Structured R1 is selected as the primary diagnostic consolidation configuration for the later controlled evidence-branch ablations.